Chunk 1 — Installation

In [31]:
# Install required Python packages for the RAG maintenance assistant

!pip install -q langchain langchain-groq langchain-community langchain-text-splitters faiss-cpu pymupdf sentence-transformers

Chunk 2 — API-Key und Modelle

In [32]:
# Load the Groq API key securely from Colab Secrets and configure the models

from google.colab import userdata
import os

GROQ_API_KEY = userdata.get("Groq_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Groq API key not found. "
        "Add it in Colab Secrets with the name 'Groq_KEY' "
        "and enable notebook access."
    )

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Model configuration
STUDENT_MODEL = "openai/gpt-oss-20b"
TEACHER_MODEL = "openai/gpt-oss-120b"

print("✅ Groq API key loaded securely from Colab Secrets.")
print(f"✅ Student model: {STUDENT_MODEL}")
print(f"✅ Teacher model: {TEACHER_MODEL}")

✅ Groq API key loaded securely from Colab Secrets.
✅ Student model: openai/gpt-oss-20b
✅ Teacher model: openai/gpt-oss-120b


Chunk 3 — Google Drive und PDFs

In [33]:
# Mount Google Drive and locate the technical handbook PDFs

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Source folder in Google Drive:
# My Drive > KI Projekte > Projekt 1 > Quellen
HANDBOOK_DIR = Path("/content/drive/MyDrive/KI Projekte/Projekt 1/Quellen")

if not HANDBOOK_DIR.exists():
    raise FileNotFoundError(f"Handbook folder not found: {HANDBOOK_DIR}")

pdf_files = sorted(HANDBOOK_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in: {HANDBOOK_DIR}")

print(f"✅ Handbook folder found: {HANDBOOK_DIR}")
print(f"✅ Found {len(pdf_files)} PDF file(s):")

for pdf_file in pdf_files:
    print(f"  📄 {pdf_file.name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Handbook folder found: /content/drive/MyDrive/KI Projekte/Projekt 1/Quellen
✅ Found 3 PDF file(s):
  📄 AEROMAX_Anleitung_R32.pdf
  📄 WPA_InfoBroschuere_Installateure.pdf
  📄 instructionandinstallationmanual_doc-00082675.pdf


Chunk 4 — RAG-System

In [34]:
# Build the RAG system: load PDFs, split text, create embeddings and a FAISS index

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("📚 Loading technical handbooks...")

documents = []

# Load every PDF page as a LangChain document
for pdf_file in HANDBOOK_DIR.glob("*.pdf"):
    loader = PyMuPDFLoader(str(pdf_file))
    documents.extend(loader.load())
    print(f"  📄 Loaded: {pdf_file.name}")

if not documents:
    raise ValueError("No documents were loaded. Check the handbook folder.")

print(f"✅ Loaded {len(documents)} PDF pages in total.")

# Split long pages into smaller chunks for semantic search
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Created {len(chunks)} text chunks.")

# Create multilingual embeddings for German technical documentation
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Build the FAISS vector index
vectorstore = FAISS.from_documents(chunks, embedding_model)

# Retrieve the four most relevant handbook passages for each question
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print("✅ FAISS vector index created.")
print("✅ Retriever ready: returns the 4 most relevant handbook passages.")

📚 Loading technical handbooks...
  📄 Loaded: instructionandinstallationmanual_doc-00082675.pdf
  📄 Loaded: WPA_InfoBroschuere_Installateure.pdf
  📄 Loaded: AEROMAX_Anleitung_R32.pdf
✅ Loaded 114 PDF pages in total.
✅ Created 306 text chunks.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ FAISS vector index created.
✅ Retriever ready: returns the 4 most relevant handbook passages.


Chunk 5 — LLM-Clients und FeedbackLogger

In [35]:
# Create Groq LLM clients and a feedback logger for DPO-style preference data

from langchain_groq import ChatGroq
from difflib import SequenceMatcher
import json
from datetime import datetime, timezone

# Student model: generates the initial technical answer
student_llm = ChatGroq(
    model=STUDENT_MODEL,
    temperature=0.3,
    api_key=GROQ_API_KEY,
)

# Teacher model: reviews and improves the student answer
teacher_llm = ChatGroq(
    model=TEACHER_MODEL,
    temperature=0.1,
    api_key=GROQ_API_KEY,
)


class FeedbackLogger:
    """
    RLAIF pipeline: stores chosen/rejected answer pairs in JSONL format.

    The output can be used for DPO-style preference tuning with tools such as
    Hugging Face TRL or Unsloth. A pair is stored only when the teacher answer
    differs sufficiently from the student answer.
    """

    def __init__(self, filepath="/content/dpo_training_data.jsonl"):
        self.filepath = filepath
        self.session_count = 0

    def log(
        self,
        prompt,
        student_answer,
        teacher_answer,
        context,
        metadata=None,
        similarity_threshold=0.85,
    ):
        """Store a preference pair when the teacher answer is sufficiently different."""

        similarity = SequenceMatcher(
            None,
            student_answer.strip(),
            teacher_answer.strip(),
        ).ratio()

        # Do not store pairs when both answers are effectively identical
        if similarity >= similarity_threshold:
            return False

        preference_pair = {
            "prompt": prompt,
            "rejected": student_answer,
            "chosen": teacher_answer,
            "context": context,
            "similarity": round(float(similarity), 4),
            "metadata": metadata or {},
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }

        with open(self.filepath, "a", encoding="utf-8") as file:
            file.write(json.dumps(preference_pair, ensure_ascii=False) + "\n")

        self.session_count += 1
        return True

    def total_saved(self):
        """Return the total number of stored preference pairs."""

        try:
            with open(self.filepath, "r", encoding="utf-8") as file:
                return sum(1 for _ in file)
        except FileNotFoundError:
            return 0


logger = FeedbackLogger()

print("✅ Student and teacher LLM clients created.")
print(f"✅ Student model: {STUDENT_MODEL}")
print(f"✅ Teacher model: {TEACHER_MODEL}")
print("✅ FeedbackLogger ready.")

✅ Student and teacher LLM clients created.
✅ Student model: openai/gpt-oss-20b
✅ Teacher model: openai/gpt-oss-120b
✅ FeedbackLogger ready.


Chunk 6 — Techniker-Agent

In [36]:
# Define the student/teacher maintenance assistant workflow

from langchain_core.messages import SystemMessage, HumanMessage

STUDENT_SYSTEM_PROMPT = (
    "You are a heating and heat-pump maintenance assistant. "
    "Answer the technician's question using only the provided handbook context. "
    "If the context does not contain enough information, state that the answer "
    "is not covered by the available handbook. "
    "Always answer in German."
)

TEACHER_SYSTEM_PROMPT = (
    "You are an experienced heating and heat-pump engineer reviewing a junior "
    "assistant's answer. Use only the provided handbook context. "
    "Correct technical errors, remove unsupported claims, and improve clarity. "
    "If the student answer is already correct and complete, keep it unchanged. "
    "Always answer in German."
)


def ask_technician(question: str, verbose: bool = False) -> dict:
    """
    Answer a technician question using RAG and a student/teacher review workflow.

    Args:
        question: The technician's question in German.
        verbose: If True, print the complete workflow output.

    Returns:
        A dictionary containing the retrieved context, student answer,
        teacher answer, and feedback status.
    """

    # 1. Retrieve the most relevant handbook passages
    relevant_chunks = retriever.invoke(question)
    context = "\n\n".join(chunk.page_content for chunk in relevant_chunks)

    if verbose:
        print("─" * 60)
        print(f"❓ FRAGE: {question}")
        print("─" * 60)
        print(f"📚 RAG: {len(relevant_chunks)} relevante Handbuch-Stellen gefunden.")
        print()

    # 2. Generate the initial answer with the student model
    student_answer = student_llm.invoke(
        [
            SystemMessage(content=STUDENT_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    f"Handbuch-Auszug:\n{context}\n\n"
                    f"Frage: {question}"
                )
            ),
        ]
    ).content

    if verbose:
        print(f"🎓 STUDENT ({STUDENT_MODEL}):")
        print(student_answer)
        print()

    # 3. Review and improve the answer with the teacher model
    teacher_answer = teacher_llm.invoke(
        [
            SystemMessage(content=TEACHER_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    f"Handbuch-Auszug:\n{context}\n\n"
                    f"Studentenantwort:\n{student_answer}\n\n"
                    f"Frage: {question}"
                )
            ),
        ]
    ).content

    if verbose:
        print(f"👨‍🏫 TEACHER ({TEACHER_MODEL}):")
        print(teacher_answer)
        print()

    # 4. Store a preference pair for potential DPO-style training
    feedback_saved = logger.log(
        prompt=question,
        student_answer=student_answer,
        teacher_answer=teacher_answer,
        context=context,
        metadata={
            "student_model": STUDENT_MODEL,
            "teacher_model": TEACHER_MODEL,
            "retrieved_chunks": len(relevant_chunks),
        },
    )

    if verbose:
        if feedback_saved:
            print(
                f"💾 RLAIF: DPO-Paar gespeichert ✓ | "
                f"DPO-Einträge gesamt: {logger.total_saved()}"
            )
        else:
            print("ℹ️ Studentenantwort war ausreichend — kein DPO-Paar gespeichert.")

    return {
        "question": question,
        "student_answer": student_answer,
        "teacher_answer": teacher_answer,
        "retrieved_chunks": relevant_chunks,
        "feedback_saved": feedback_saved,
    }

Chunk 7 — Demo

In [37]:
# Run three representative maintenance questions through the RAG pipeline

demo_questions = [
    "Wie tausche ich das Druckventil DV-12 aus? Welche Werkzeuge brauche ich und worauf muss ich achten?",
    "Was bedeutet der Fehlercode E07?",
    "Wie oft muss das Kompressoröl K1 gewechselt werden?",
]

results = []

for question in demo_questions:
    result = ask_technician(question, verbose=True)
    results.append(result)
    print()

────────────────────────────────────────────────────────────
❓ FRAGE: Wie tausche ich das Druckventil DV-12 aus? Welche Werkzeuge brauche ich und worauf muss ich achten?
────────────────────────────────────────────────────────────
📚 RAG: 4 relevante Handbuch-Stellen gefunden.

🎓 STUDENT (openai/gpt-oss-20b):
Die im Handbuch enthaltenen Informationen beziehen sich ausschließlich auf Kondensatabläufe, Steckverbinder und allgemeine Installationsmaßnahmen. Ein Austausch des Druckventils DV‑12, die benötigten Werkzeuge und die zu beachtenden Punkte sind dort nicht beschrieben. Daher kann ich dazu keine Auskunft geben.

👨‍🏫 TEACHER (openai/gpt-oss-120b):
Der vorliegende Handbuch‑Auszug enthält ausschließlich Anweisungen zum Kondensatablauf, zu Steckverbindern und zu allgemeinen Installationsarbeiten. Informationen zum Druckventil DV‑12, zu den dafür erforderlichen Werkzeugen oder zu speziellen Vorgehensweisen beim Austausch dieses Bauteils sind darin nicht enthalten.  

Daher lässt sich aus 

Chunk 8 - DPO-Trainingsdaten ansehen

In [29]:
dpo_file = Path("/content/dpo_training_data.jsonl")

if not dpo_file.exists() or dpo_file.stat().st_size == 0:
    print("ℹ️  Noch keine DPO-Daten. Bitte zuerst Cell 7 ausführen.")
else:
    eintraege = [json.loads(line) for line in dpo_file.open(encoding="utf-8")]
    print(f"📋 {len(eintraege)} DPO-Paare gespeichert:\n")
    for i, e in enumerate(eintraege, 1):
        print(f"{'─'*55}")
        print(f"Paar #{i}  |  Similarity: {e['similarity']}  |  {e['timestamp'][:16]}")
        print(f"PROMPT:   {e['prompt']}")
        print(f"REJECTED: {e['rejected'][:200]}...")
        print(f"CHOSEN:   {e['chosen'][:200]}...")
        print()
    print("💡 Bereit für DPO-Training mit HuggingFace TRL oder Unsloth!")

📋 3 DPO-Paare gespeichert:

───────────────────────────────────────────────────────
Paar #1  |  Similarity: 0.0  |  2026-10-19T16:53
PROMPT:   Wie tausche ich das Druckventil DV-12 aus? Welche Werkzeuge brauche ich und worauf muss ich achten?
REJECTED: ...
CHOSEN:   **Austausch des Druckventils DV‑12 – Schritt‑für‑Schritt‑Anleitung**

| **Arbeitsschritt** | **Beschreibung** | **Werkzeuge / Hilfsmittel** |
|--------------------|-------------------|----------------...

───────────────────────────────────────────────────────
Paar #2  |  Similarity: 0.1341  |  2026-10-19T16:53
PROMPT:   Was bedeutet Fehlercode E07 und was muss ich als Techniker sofort tun?
REJECTED: **Fehlercode E07 – nicht im Handbuch‑Auszug enthalten**

Der von Ihnen genannte Fehlercode **E07** ist im bereitgestellten Auszug des Handbuchs nicht aufgeführt.  
Daher lässt sich aus den vorhandenen...
CHOSEN:   **Fehlercode E07 ist im vorliegenden Handbuch‑Auszug nicht aufgeführt.**  
Daher kann aus den vorhandenen Informati

Chunk 9 - Live-Demo

In [30]:
print("🎤 LIVE-DEMO – Stell eine eigene Techniker-Frage!")
print("   (Enter ohne Text drücken zum Beenden)\n")

while True:
    frage = input("❓ Deine Frage: ").strip()
    if not frage:
        print(f"\nDemo beendet.  Gesamt gespeicherte DPO-Paare: {logger.total_saved()}")
        break
    ask_technician(frage, verbose=True)

🎤 LIVE-DEMO – Stell eine eigene Techniker-Frage!
   (Enter ohne Text drücken zum Beenden)



KeyboardInterrupt: Interrupted by user